# L0: identify inertia and viscous damping

This notebook is the inspectable computation surface for the first SysID lesson. The plant is `J*qdd + b*qd = u`, with known applied torque and ideal `t`, `u`, `q`, and `qd` observations. The Oracle parameters are used only to generate and evaluate the lesson; fitting receives observations, public bounds, and a deliberately wrong Initial-model parameter set. The configuration key remains `nominal` for compatibility.

## Explain -> show -> change

The fit uses a chirp and validation uses a separate multisine. Run the cells once to see the True system (Oracle), Initial model, and Identified model. Then change one of the Initial-model values below and rerun the experiment cell. The edit changes the orange baseline and the optimizer starting point; when the fit succeeds, the identified parameters should still return to the Oracle values.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'synthetic').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from synthetic.l0_inertia_damping import (
    Params, load_config, plot_run, run_l0, run_summary,
)

config = load_config(ROOT / 'synthetic' / 'l0_config.json')
config.version

In [ ]:
# Expose one simple learner control: the Initial-model parameters.
nominal_inertia = config.nominal.inertia
nominal_damping = config.nominal.damping
lesson_config = replace(
    config,
    nominal=Params(inertia=nominal_inertia, damping=nominal_damping),
)
run = run_l0(lesson_config)
summary = run_summary(run)
summary['fit'], summary['validation']

In [ ]:
report_dir = ROOT / 'artifacts' / 'l0_inertia_damping_notebook'
plot_path = plot_run(run, report_dir / 'report.png')
from IPython.display import Image, display
display(Image(filename=str(plot_path)))

## Questions

- Which parts of the Initial-model trajectory reveal that its inertia and damping are wrong?
- Why is the validation curve evidence rather than another fitting target?
- Which assumptions would stop this result from transferring to a real servo? Name at least two omitted effects, such as torque calibration/controller behavior, delay, saturation, friction, compliance, contact, or sensor noise.